# 05. 최종 모델·평가

- 입력: `data/processed/X_*.csv`, `y_*.csv` (03에서 저장), `results/best_config.json` (04에서 저장)
- 하는 일: 최적 조합으로 train 데이터로 학습 → **test 데이터로 1회만 평가** (MSE, RMSE, MAE) → 예측 vs 실제 그래프
- 출력: 없음 (최종 성능 표와 그래프)

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트(notebooks/의 상위 폴더)를 import 경로에 추가해서 src 패키지를 사용
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import BEST_CONFIG_PATH
from src.data import load_splits
from src.model import build_model, plot_history

X_train_np, X_val_np, X_test_np, y_train_np, y_val_np, y_test_np = load_splits()

with open(BEST_CONFIG_PATH, encoding='utf-8') as f:
    best_config = json.load(f)
best_config['hidden_units'] = tuple(best_config['hidden_units'])  # json에는 리스트로 저장됨

In [ ]:
print('최종 하이퍼파라미터:', best_config)

final_model = build_model(X_train_np.shape[1], best_config['hidden_units'], best_config['activation'], best_config['learning_rate'])
use_log = best_config['target'] == 'log1p'
final_history = final_model.fit(
    X_train_np, np.log1p(y_train_np) if use_log else y_train_np,
    validation_data=(X_val_np, np.log1p(y_val_np) if use_log else y_val_np),
    epochs=best_config['epochs'], batch_size=best_config['batch_size'], verbose=0,
)
final_model.summary()

In [ ]:
# 최종 모델 학습 곡선
plot_history(final_history, 'Final model')

In [ ]:
# test 데이터로 1회 평가
def predict(model, X):
    pred = model.predict(X, verbose=0).ravel()
    return np.expm1(pred) if use_log else pred

final_scores = {}
for name, X_part, y_part in [('validation', X_val_np, y_val_np), ('test', X_test_np, y_test_np)]:
    pred = predict(final_model, X_part)
    mse = float(np.mean((pred - y_part) ** 2))
    final_scores[name] = {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': float(np.mean(np.abs(pred - y_part)))}

test_pred = predict(final_model, X_test_np)
print('test 음수 예측 개수:', (test_pred < 0).sum())
pd.DataFrame(final_scores).T.round(1)

In [ ]:
# 예측 vs 실제 (test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_np, test_pred, s=5, alpha=0.4)
lim = [0, max(y_test_np.max(), test_pred.max()) * 1.05]
ax.plot(lim, lim, color='red', linestyle='--', label='y = x')
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel('Actual rented bike count')
ax.set_ylabel('Predicted rented bike count')
ax.set_title('Test: predicted vs actual')
ax.legend()
ax.grid(alpha=0.3)
plt.show()